In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.model_selection import train_test_split # Import for splitting data into training and testing sets
from sklearn.compose import ColumnTransformer # Import for applying transformers to specific columns
from sklearn.impute import SimpleImputer # Import for handling missing values
from sklearn.preprocessing import OneHotEncoder # Import for one-hot encoding categorical features
from sklearn.preprocessing import MinMaxScaler # Import for scaling features to a specified range
from sklearn.pipeline import Pipeline, make_pipeline # Import for creating and managing a sequence of data transformations and model training
from sklearn.tree import DecisionTreeClassifier # Import the Decision Tree Classifier model
from sklearn.feature_selection import SelectKBest, chi2 # Import for selecting top features based on chi-squared test

In [ ]:
df = pd.read_csv('/content/train.csv')

In [ ]:
df.head(1)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,NaN,S


## Lets Plan

In [ ]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'], inplace = True) # Drop specified columns from the DataFrame in-place as they are not needed for modeling

In [ ]:
# Step 1 -> train/test/split
# Split the DataFrame into training and testing sets for features (X) and target (y)
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']), # Features (all columns except 'Survived')
                                                    df['Survived'], # Target variable ('Survived')
                                                    test_size = 0.2, # Allocate 20% of data for testing
                                                    random_state = 42) # Set a random seed for reproducibility

In [ ]:
X_train.head() # Display the first few rows of the training features to verify the split

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [ ]:
# imputation transformer
# Create a ColumnTransformer to handle missing values
trf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), [2]), # Impute missing 'Age' (column index 2) with the mean
    ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]) # Impute missing 'Embarked' (column index 6) with the most frequent value
], remainder='passthrough') # Keep other columns as they are

In [ ]:
# one hot encoding
# Create a ColumnTransformer for one-hot encoding categorical features
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(handle_unknown='ignore', sparse_output=False), [1,6]) # One-hot encode 'Sex' (index 1) and 'Embarked' (index 6)
], remainder='passthrough') # Keep other columns as they are

In [ ]:
# Scaling
# Create a ColumnTransformer for feature scaling
trf3 = ColumnTransformer([
    ('scale', MinMaxScaler(), slice(0,10)) # Apply MinMaxScaler to all numerical features (columns 0-9)
])

In [ ]:
# Feature Selection
# Create a SelectKBest transformer to select the top 5 features using the chi-squared test
trf4 = SelectKBest(score_func=chi2, k=5)

In [ ]:
# train the model
# Initialize a Decision Tree Classifier model
trf5 = DecisionTreeClassifier()

## Create Pipeline

In [ ]:
pipe = Pipeline([
    ('trf1', trf1), # Step 1: Imputation of missing values
    ('trf2', trf2), # Step 2: One-hot encoding of categorical features
    ('trf3', trf3), # Step 3: Feature scaling
    ('trf4', trf4), # Step 4: Feature selection
    ('trf5', trf5) # Step 5: Decision Tree Classifier model training
])

# Pipeline Vs make_pipeline

In [ ]:
# Altering syntax
# Alternative way to create a pipeline: pipe = make_pipeline(trf1, trf2, trf3, trf4, trf5)

In [ ]:
# train
# Fit the pipeline on the training data (X_train and y_train)
pipe.fit(X_train, y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=5,
                             score_func=<function chi2 at 0x7bbbaecc7ba0>)),
                ('trf5', DecisionTreeClassifier())])

In [ ]:
# display pipeline
from sklearn import set_config # Import set_config to change scikit-learn display options
set_config(display='diagram') # Configure scikit-learn to display pipelines as diagrams

In [ ]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_ # Access the statistics (e.g., mean) used by the first transformer (impute_age) in the pipeline

array([29.49884615])

In [ ]:
# predict
# Make predictions on the test set using the trained pipeline
y_pred = pipe.predict(X_test)

In [ ]:
y_pred # Display the predicted values for the test set

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0])

In [ ]:
from sklearn.metrics import accuracy_score # Import the accuracy_score metric for model evaluation
accuracy_score(y_test, y_pred) # Calculate and display the accuracy of the model on the test set

0.6256983240223464

## CrossValidation using Pipeline

In [ ]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score # Import cross_val_score for cross-validation
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean() # Perform 5-fold cross-validation on the training data and calculate the mean accuracy

np.float64(0.6391214419383433)

## GridSearch using pipeline

In [ ]:
# gridsearchcv
# Define the parameter grid for GridSearchCV
params = {
    'trf5__max_depth': [1,2,3,4,5,None] # Explore different values for the 'max_depth' parameter of the Decision Tree Classifier (trf5)
}

In [ ]:
from sklearn.model_selection import GridSearchCV # Import GridSearchCV for hyperparameter tuning
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy') # Initialize GridSearchCV with the pipeline, parameter grid, 5-fold cross-validation, and accuracy scoring
grid.fit(X_train, y_train) # Fit GridSearchCV to the training data to find the best hyperparameters

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('trf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf4',
                                        SelectKBest(k=5,
                                                    score_func=<function chi2 at 0x7bbbaecc7ba0>)),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [ ]:
grid.best_score_ # Display the best cross-validation score found by GridSearchCV

np.float64(0.6391214419383433)

In [ ]:
grid.best_params_ # Display the best hyperparameters found by GridSearchCV

{'trf5__max_depth': 2}